# K-IFRS Enhanced RAG Pipeline Test

**새 모듈 통합 테스트**: Query Router → Authority Filter → Hybrid Retrieval → Rerank → Authority Boost → Cross-Ref Expansion → Term Definition Injection → LLM Generate

각 단계의 결과를 **본문(main) / 적용지침(ag) / 결론도출근거(bc) / 사례(ie)** 구분으로 시각화한다.

In [1]:
import os
import time

from dotenv import load_dotenv
from IPython.display import display, Markdown, HTML
from langchain_core.documents import Document
from langchain_upstage import UpstageEmbeddings
from qdrant_client import QdrantClient

from search import (
    QDRANT_PATH, CHUNKS_DIR, CHILD_COLLECTION, PARENT_COLLECTION, MODEL_NAME,
    QdrantDenseRetriever, load_child_documents, kiwi_tokenize,
    fetch_siblings, search_with_parent,
    get_authority_filter, AUTHORITY_FILTERS,
    get_reranker,
    classify_query, apply_authority_boost, QueryType, QueryPlan,
    resolve_cross_refs,
    inject_term_definitions,
)
from eval.evaluator import load_test_cases, evaluate_retrieval, print_eval_summary, EvalResult

load_dotenv()
print("imports complete")

/home/shin/Study/_database/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


imports complete


## 0. Section-Type Color Legend

| section_type | 구분 | 색상 |
|---|---|---|
| main | 본문 | 🔵 blue |
| ag | 적용지침 | 🟢 green |
| bc | 결론도출근거 | 🟠 orange |
| ie | 사례 | 🟣 purple |
| glossary | 용어정의 | ⚫ grey |
| xref | 교차참조확장 | 🔷 teal |

In [2]:
SECTION_COLORS = {
    "main": "#2196F3",
    "ag":   "#4CAF50",
    "bc":   "#FF9800",
    "ie":   "#9C27B0",
}
SECTION_LABELS = {
    "main": "본문", "ag": "적용지침", "bc": "결론도출근거", "ie": "사례",
}

def section_badge(section_type: str, extra: str = "") -> str:
    color = SECTION_COLORS.get(section_type, "#607D8B")
    label = SECTION_LABELS.get(section_type, section_type)
    if extra:
        label = f"{label} | {extra}"
    return f'<span style="background:{color};color:#fff;padding:2px 8px;border-radius:4px;font-size:0.85em;">{label}</span>'

def display_docs(docs: list[Document], title: str = "", max_content: int = 200):
    parts = [f"<h4>{title}</h4>"] if title else []
    for i, doc in enumerate(docs, 1):
        m = doc.metadata
        st = m.get("section_type", "?")
        extras = []
        if m.get("is_glossary"):
            extras.append("glossary")
        if m.get("fetched_by_xref"):
            extras.append(f"xref:{m.get('xref_source','')}")
        score = m.get("rerank_score")
        if score is not None:
            extras.append(f"score={score:.4f}")

        badge = section_badge(st, " | ".join(extras))
        std = m.get("standard_id", "?")
        para = m.get("para_number", "?")
        chunk_id = m.get("chunk_id", "")
        preview = doc.page_content[:max_content].replace("\n", " ")

        color = SECTION_COLORS.get(st, "#607D8B")
        parts.append(
            f'<div style="border-left:4px solid {color};padding:6px 12px;margin:4px 0;">'
            f'<b>[{i}]</b> {badge} {std} 문단 {para} '
            f'<code style="font-size:0.8em;">{chunk_id}</code><br/>'
            f'<span style="color:#555;">{preview}...</span></div>'
        )
    display(HTML("".join(parts)))

def section_summary(docs: list[Document]) -> dict[str, int]:
    counts: dict[str, int] = {}
    for d in docs:
        st = d.metadata.get("section_type", "unknown")
        counts[st] = counts.get(st, 0) + 1
    return counts

print("display helpers defined")

display helpers defined


## 1. Initialize Clients

In [3]:
print("[1/3] Upstage embeddings...")
embeddings = UpstageEmbeddings(
    model=MODEL_NAME,
    upstage_api_key=os.getenv("UPSTAGE_API_KEY"),
)

print("[2/3] Qdrant client...")
client = QdrantClient(path=QDRANT_PATH)
print(f"  Child: {client.count(CHILD_COLLECTION).count} points")
print(f"  Parent: {client.count(PARENT_COLLECTION).count} points")

print("[3/3] Reranker...")
reranker = get_reranker(reranker_type='local')
print(f"  Type: {type(reranker).__name__}")
print("init complete")

[1/3] Upstage embeddings...
[2/3] Qdrant client...
  Child: 12313 points
  Parent: 2231 points
[3/3] Reranker...
  Type: LocalReranker
init complete


## 2. Stage 1: Query Router

`classify_query(query)` → `QueryPlan` (query_type, filter, k, rerank_n, authority_boost)

규칙 기반 분류기로 5가지 유형별 대표 쿼리를 테스트한다.

In [4]:
ROUTER_TEST_QUERIES = [
    "유형자산의 감가상각 방법에는 어떤 것이 있는가?",         # normative
    "왜 IFRS는 공정가치 측정을 요구하는가?",                # interpretive
    "리스 회계처리 사례를 보여줘",                          # example
    "제1016호 문단 62",                                    # citation
    "충당부채와 우발부채의 차이는?",                        # comparative
    "금융자산의 분류 기준은 무엇인가?",                     # normative
]

lines = ["| Query | Type | Filter | K | Rerank N | Auth Boost | Direct IDs |",
         "|---|---|---|---|---|---|---|"]
plans = {}
for q in ROUTER_TEST_QUERIES:
    plan = classify_query(q)
    plans[q] = plan
    short_q = q if len(q) <= 30 else q[:27] + "..."
    filter_str = "normative" if plan.query_filter and "main" in str(plan.query_filter) else (
        "ie" if plan.query_filter and "ie" in str(plan.query_filter) else "full"
    )
    ids_str = ", ".join(plan.direct_lookup_ids) if plan.direct_lookup_ids else "-"
    lines.append(
        f"| {short_q} | **{plan.query_type.value}** | {filter_str} | {plan.retrieval_k} | {plan.rerank_n} | {plan.authority_boost} | `{ids_str}` |"
    )
display(Markdown("\n".join(lines)))

| Query | Type | Filter | K | Rerank N | Auth Boost | Direct IDs |
|---|---|---|---|---|---|---|
| 유형자산의 감가상각 방법에는 어떤 것이 있는가? | **normative** | normative | 30 | 10 | True | `-` |
| 왜 IFRS는 공정가치 측정을 요구하는가? | **interpretive** | full | 30 | 10 | False | `-` |
| 리스 회계처리 사례를 보여줘 | **example** | ie | 20 | 10 | False | `-` |
| 제1016호 문단 62 | **citation** | full | 5 | 5 | False | `KIFRS1016_main_62` |
| 충당부채와 우발부채의 차이는? | **comparative** | full | 40 | 15 | True | `-` |
| 금융자산의 분류 기준은 무엇인가? | **normative** | normative | 30 | 10 | True | `-` |

## 3. Stage 2: Authority Filter + Dense Retrieval

QueryPlan의 필터를 `QdrantDenseRetriever`에 적용하여 section_type 분포 변화를 관찰한다.

In [5]:
TEST_QUERY = "유형자산의 감가상각 방법에는 어떤 것이 있는가?"
plan = plans[TEST_QUERY]

# Filtered retrieval (normative -> main+ag only)
retriever_filtered = QdrantDenseRetriever(
    client=client, embeddings=embeddings,
    collection_name=CHILD_COLLECTION,
    k=plan.retrieval_k,
    query_filter=plan.query_filter,
)
t0 = time.time()
docs_filtered = retriever_filtered.invoke(TEST_QUERY)
elapsed = time.time() - t0

print(f"Query: {TEST_QUERY}")
print(f"Filter: {plan.query_type.value} | Retrieved: {len(docs_filtered)} docs ({elapsed:.2f}s)")
print(f"Section distribution: {section_summary(docs_filtered)}")
display_docs(docs_filtered[:10], title=f"Filtered retrieval (top 10 of {len(docs_filtered)})")

Query: 유형자산의 감가상각 방법에는 어떤 것이 있는가?
Filter: normative | Retrieved: 30 docs (1.05s)
Section distribution: {'main': 29, 'ag': 1}


In [6]:
# Unfiltered comparison
retriever_full = QdrantDenseRetriever(
    client=client, embeddings=embeddings,
    collection_name=CHILD_COLLECTION,
    k=plan.retrieval_k,
    query_filter=None,
)
docs_full = retriever_full.invoke(TEST_QUERY)

print(f"Unfiltered retrieval: {len(docs_full)} docs")
print(f"Section distribution: {section_summary(docs_full)}")
display_docs(docs_full[:10], title="Unfiltered retrieval (top 10)")

# Side-by-side comparison
lines = ["| Mode | main | ag | bc | ie |", "|---|---|---|---|---|"]
for label, docs in [("Filtered (normative)", docs_filtered), ("Full", docs_full)]:
    s = section_summary(docs)
    lines.append(f"| {label} | {s.get('main',0)} | {s.get('ag',0)} | {s.get('bc',0)} | {s.get('ie',0)} |")
display(Markdown("\n".join(lines)))

Unfiltered retrieval: 30 docs
Section distribution: {'main': 22, 'bc': 8}


| Mode | main | ag | bc | ie |
|---|---|---|---|---|
| Filtered (normative) | 29 | 1 | 0 | 0 |
| Full | 22 | 0 | 8 | 0 |

## 4. Stage 3: Rerank

Cross-encoder reranker를 적용하여 의미적 관련도 기준으로 재정렬한다.

In [7]:
t0 = time.time()
docs_reranked = reranker.rerank(TEST_QUERY, docs_filtered, top_n=plan.rerank_n)
elapsed = time.time() - t0

print(f"Reranked: {len(docs_filtered)} -> {len(docs_reranked)} docs ({elapsed:.2f}s)")
print(f"Section distribution after rerank: {section_summary(docs_reranked)}")
print(f"Score range: {docs_reranked[-1].metadata['rerank_score']:.4f} ~ {docs_reranked[0].metadata['rerank_score']:.4f}")
display_docs(docs_reranked, title=f"After Rerank (top {plan.rerank_n})")

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Reranked: 30 -> 10 docs (4.00s)
Section distribution after rerank: {'main': 10}
Score range: 0.9793 ~ 0.9999


## 5. Stage 4: Authority Boost

`apply_authority_boost()` — bc/ie 문서의 rerank score에 감쇠 계수(0.85)를 적용한다.
효과를 보기 위해 comparative 쿼리(full filter, authority_boost=True)로 테스트한다.

In [8]:
BOOST_QUERY = "충당부채와 우발부채의 차이는?"
boost_plan = plans[BOOST_QUERY]
print(f"Query: {BOOST_QUERY}")
print(f"Type: {boost_plan.query_type.value}, authority_boost={boost_plan.authority_boost}")

# Retrieve + rerank with full filter
retriever_boost = QdrantDenseRetriever(
    client=client, embeddings=embeddings,
    collection_name=CHILD_COLLECTION,
    k=boost_plan.retrieval_k,
    query_filter=boost_plan.query_filter,
)
docs_boost_raw = retriever_boost.invoke(BOOST_QUERY)
docs_boost_reranked = reranker.rerank(BOOST_QUERY, docs_boost_raw, top_n=boost_plan.rerank_n)

print(f"\nBefore authority boost: {section_summary(docs_boost_reranked)}")
display_docs(docs_boost_reranked, title="Before Authority Boost")

# Apply boost
docs_boosted = apply_authority_boost(docs_boost_reranked, boost_factor=0.85)
print(f"\nAfter authority boost: {section_summary(docs_boosted)}")
display_docs(docs_boosted, title="After Authority Boost (bc/ie scores * 0.85)")

# Show score delta for any bc/ie docs
bc_ie_docs = [d for d in docs_boost_reranked if d.metadata.get("section_type") in ("bc", "ie")]
if bc_ie_docs:
    print("\nbc/ie score changes:")
    for d in bc_ie_docs:
        orig = d.metadata["rerank_score"]
        print(f"  {d.metadata['chunk_id']}: {orig:.4f} -> {orig*0.85:.4f}")
else:
    print("\nNo bc/ie docs in results to demonstrate boost on.")

Query: 충당부채와 우발부채의 차이는?
Type: comparative, authority_boost=True

Before authority boost: {'main': 11, 'ag': 2, 'bc': 1, 'ie': 1}



After authority boost: {'main': 11, 'ag': 2, 'bc': 1, 'ie': 1}



bc/ie score changes:
  KIFRS1037_bc_unk_35: 0.9900 -> 0.8415
  KIFRS1037_ie_unk_28: 0.9240 -> 0.7854


## 6. Stage 5: Cross-Reference Expansion

`resolve_cross_refs()` — 검색 결과의 `cross_refs` 메타데이터를 분석하여 참조된 문단을 Qdrant에서 자동 fetch한다.

In [9]:
XREF_QUERY = "금융자산의 분류 기준은 무엇인가?"
xref_plan = classify_query(XREF_QUERY)
print(f"Query: {XREF_QUERY}")
print(f"Type: {xref_plan.query_type.value}")

retriever_xref = QdrantDenseRetriever(
    client=client, embeddings=embeddings,
    collection_name=CHILD_COLLECTION,
    k=xref_plan.retrieval_k,
    query_filter=xref_plan.query_filter,
)
docs_xref_raw = retriever_xref.invoke(XREF_QUERY)
docs_xref_reranked = reranker.rerank(XREF_QUERY, docs_xref_raw, top_n=xref_plan.rerank_n)

# Show cross_refs metadata before expansion
print(f"\nReranked docs: {len(docs_xref_reranked)}")
for d in docs_xref_reranked:
    refs = d.metadata.get("cross_refs", [])
    if refs:
        print(f"  {d.metadata['chunk_id']} -> cross_refs: {refs[:5]}")

# Expand
t0 = time.time()
docs_expanded = resolve_cross_refs(docs_xref_reranked, client, max_expansion=10)
elapsed = time.time() - t0

n_new = len(docs_expanded) - len(docs_xref_reranked)
print(f"\nExpanded: {len(docs_xref_reranked)} + {n_new} xref docs = {len(docs_expanded)} total ({elapsed:.2f}s)")

# Display only the new xref docs
xref_docs = [d for d in docs_expanded if d.metadata.get("fetched_by_xref")]
if xref_docs:
    display_docs(xref_docs, title=f"Cross-Ref Expanded Documents ({len(xref_docs)})")
else:
    print("No cross-reference documents were resolved (refs may not exist in DB).")

Query: 금융자산의 분류 기준은 무엇인가?
Type: normative

Reranked docs: 10
  KIFRS1109_ag_unk_9 -> cross_refs: ['B4.1.1', 'B4.1.10', 'B4.1.10A', 'B4.1.11', 'B4.1.12']
  KIFRS1109_main_7.2.9 -> cross_refs: ['문단4.1.5']
  KIFRS1107_main_unk_5 -> cross_refs: ['문단7.2.2', '제1039호', '제1109호']
  KIFRS1032_ag_AG14J -> cross_refs: ['문단16A', '문단16C']
  KIFRS1109_main_7.2.32 -> cross_refs: ['문단4.1.5', '문단4.2.2']
  KIFRS1109_ag_unk_24 -> cross_refs: ['B3.1.5', '문단5.7.4']

Expanded: 10 + 0 xref docs = 10 total (0.00s)
No cross-reference documents were resolved (refs may not exist in DB).


## 7. Stage 6: Term Definition Injection

`inject_term_definitions()` — 검색 결과에서 빈도 높은 기준서의 용어정의(Appendix A) 청크를 컨텍스트 선두에 주입한다.

In [10]:
t0 = time.time()
docs_with_terms = inject_term_definitions(docs_expanded, client, max_definitions=3)
elapsed = time.time() - t0

n_glossary = sum(1 for d in docs_with_terms if d.metadata.get("is_glossary"))
print(f"Injected {n_glossary} glossary doc(s) ({elapsed:.2f}s)")
print(f"Total context: {len(docs_with_terms)} docs")

glossary_docs = [d for d in docs_with_terms if d.metadata.get("is_glossary")]
if glossary_docs:
    display_docs(glossary_docs, title="Injected Term Definitions", max_content=300)

# Final context composition
summary = section_summary(docs_with_terms)
print(f"\nFinal context composition: {summary}")

Injected 0 glossary doc(s) (0.00s)
Total context: 10 docs

Final context composition: {'main': 6, 'ag': 4}


## 8. Full Pipeline: End-to-End

전체 6단계를 하나의 함수로 통합: Route → Filter → Retrieve → Rerank → Authority Boost → XRef → Terms

In [11]:
def run_pipeline(
    query: str,
    client: QdrantClient,
    embeddings: UpstageEmbeddings,
    reranker,
    verbose: bool = True,
) -> tuple[list[Document], dict]:
    """Execute the full RAG retrieval pipeline. Returns (final_docs, stage_log)."""
    log = {}

    # Stage 1: Query Router
    plan = classify_query(query)
    log["plan"] = plan
    if verbose:
        print(f"  [Route] type={plan.query_type.value}, k={plan.retrieval_k}, "
              f"rerank_n={plan.rerank_n}, boost={plan.authority_boost}")

    # Stage 2: Dense Retrieval with authority filter
    if plan.skip_vector_search:
        if verbose:
            print(f"  [Retrieve] SKIP vector search, direct IDs: {plan.direct_lookup_ids}")
        docs_retrieved = []
    else:
        retriever = QdrantDenseRetriever(
            client=client, embeddings=embeddings,
            collection_name=CHILD_COLLECTION,
            k=plan.retrieval_k,
            query_filter=plan.query_filter,
        )
        docs_retrieved = retriever.invoke(query)
    log["retrieved"] = docs_retrieved
    if verbose:
        print(f"  [Retrieve] {len(docs_retrieved)} docs | {section_summary(docs_retrieved)}")

    # Stage 3: Rerank
    docs_reranked = reranker.rerank(query, docs_retrieved, top_n=plan.rerank_n)
    log["reranked"] = docs_reranked
    if verbose:
        print(f"  [Rerank] {len(docs_retrieved)} -> {len(docs_reranked)} docs")

    # Stage 4: Authority Boost
    if plan.authority_boost:
        docs_boosted = apply_authority_boost(docs_reranked)
    else:
        docs_boosted = docs_reranked
    log["boosted"] = docs_boosted
    if verbose and plan.authority_boost:
        print(f"  [Boost] applied (factor=0.85)")

    # Stage 5: Cross-Ref Expansion
    docs_xref = resolve_cross_refs(docs_boosted, client, max_expansion=10)
    n_xref = len(docs_xref) - len(docs_boosted)
    log["xref_expanded"] = docs_xref
    if verbose:
        print(f"  [XRef] +{n_xref} docs | total={len(docs_xref)}")

    # Stage 6: Term Definition Injection
    docs_final = inject_term_definitions(docs_xref, client, max_definitions=3)
    n_glossary = sum(1 for d in docs_final if d.metadata.get("is_glossary"))
    log["final"] = docs_final
    if verbose:
        print(f"  [Terms] +{n_glossary} glossary | total={len(docs_final)}")

    return docs_final, log

print("run_pipeline defined")

run_pipeline defined


In [12]:
FULL_QUERY = "유형자산의 감가상각 방법에는 어떤 것이 있는가?"
print(f"Query: {FULL_QUERY}\n")

t0 = time.time()
final_docs, stage_log = run_pipeline(FULL_QUERY, client, embeddings, reranker)
elapsed = time.time() - t0

print(f"\nTotal pipeline time: {elapsed:.2f}s")
print(f"Final context: {len(final_docs)} docs | {section_summary(final_docs)}")
display_docs(final_docs, title="Full Pipeline Output")

Query: 유형자산의 감가상각 방법에는 어떤 것이 있는가?

  [Route] type=normative, k=30, rerank_n=10, boost=True
  [Retrieve] 30 docs | {'main': 29, 'ag': 1}
  [Rerank] 30 -> 10 docs
  [Boost] applied (factor=0.85)
  [XRef] +0 docs | total=10
  [Terms] +1 glossary | total=11

Total pipeline time: 2.37s
Final context: 11 docs | {'main': 11}


## 9. Pipeline Comparison Across Query Types

5가지 QueryType 대표 쿼리로 파이프라인을 실행하고, 단계별 수량과 section_type 분포를 비교한다.

In [13]:
COMPARISON_QUERIES = {
    "normative":    "유형자산의 감가상각 방법에는 어떤 것이 있는가?",
    "interpretive": "왜 IFRS는 공정가치 측정을 요구하는가?",
    "example":      "리스 회계처리 사례를 보여줘",
    "citation":     "제1016호 문단 62",
    "comparative":  "충당부채와 우발부채의 차이는?",
}

comparison_results = {}
for qtype, query in COMPARISON_QUERIES.items():
    print(f"\n{'='*60}")
    print(f"[{qtype.upper()}] {query}")
    print('='*60)
    docs, log = run_pipeline(query, client, embeddings, reranker)
    comparison_results[qtype] = {"docs": docs, "log": log, "query": query}

# Summary table
lines = [
    "| Query Type | #Retrieved | #Reranked | #XRef | #Glossary | #Final | main | ag | bc | ie |",
    "|---|---|---|---|---|---|---|---|---|---|",
]
for qtype, res in comparison_results.items():
    log = res["log"]
    n_ret = len(log.get("retrieved", []))
    n_rer = len(log.get("reranked", []))
    n_xref = len(log.get("xref_expanded", [])) - n_rer
    n_gloss = sum(1 for d in log.get("final", []) if d.metadata.get("is_glossary"))
    n_final = len(log.get("final", []))
    s = section_summary(log.get("final", []))
    lines.append(
        f"| **{qtype}** | {n_ret} | {n_rer} | +{max(n_xref,0)} | +{n_gloss} | {n_final} "
        f"| {s.get('main',0)} | {s.get('ag',0)} | {s.get('bc',0)} | {s.get('ie',0)} |"
    )
display(Markdown("\n".join(lines)))


[NORMATIVE] 유형자산의 감가상각 방법에는 어떤 것이 있는가?
  [Route] type=normative, k=30, rerank_n=10, boost=True
  [Retrieve] 30 docs | {'main': 29, 'ag': 1}
  [Rerank] 30 -> 10 docs
  [Boost] applied (factor=0.85)
  [XRef] +0 docs | total=10
  [Terms] +1 glossary | total=11

[INTERPRETIVE] 왜 IFRS는 공정가치 측정을 요구하는가?
  [Route] type=interpretive, k=30, rerank_n=10, boost=False
  [Retrieve] 30 docs | {'bc': 30}
  [Rerank] 30 -> 10 docs
  [XRef] +7 docs | total=17
  [Terms] +2 glossary | total=19

[EXAMPLE] 리스 회계처리 사례를 보여줘
  [Route] type=example, k=20, rerank_n=10, boost=False
  [Retrieve] 20 docs | {'ie': 20}
  [Rerank] 20 -> 10 docs
  [XRef] +3 docs | total=13
  [Terms] +2 glossary | total=15

[CITATION] 제1016호 문단 62
  [Route] type=citation, k=5, rerank_n=5, boost=False
  [Retrieve] SKIP vector search, direct IDs: ['KIFRS1016_main_62']
  [Retrieve] 0 docs | {}
  [Rerank] 0 -> 0 docs
  [XRef] +0 docs | total=0
  [Terms] +0 glossary | total=0

[COMPARATIVE] 충당부채와 우발부채의 차이는?
  [Route] type=comparative, k=40, 

| Query Type | #Retrieved | #Reranked | #XRef | #Glossary | #Final | main | ag | bc | ie |
|---|---|---|---|---|---|---|---|---|---|
| **normative** | 30 | 10 | +0 | +1 | 11 | 11 | 0 | 0 | 0 |
| **interpretive** | 30 | 10 | +7 | +2 | 19 | 2 | 0 | 17 | 0 |
| **example** | 20 | 10 | +3 | +2 | 15 | 4 | 0 | 0 | 11 |
| **citation** | 0 | 0 | +0 | +0 | 0 | 0 | 0 | 0 | 0 |
| **comparative** | 40 | 15 | +0 | +1 | 16 | 12 | 2 | 1 | 1 |

## 10. Quantitative Evaluation

`eval/test_cases.json` (20개 테스트 케이스)와 `evaluate_retrieval()`로 5가지 지표를 정량 측정한다:
- **DRM** (Document-level Retrieval Mismatch) — 낮을수록 좋음
- **XRef Coverage** — 교차참조 포함률
- **Authority Accuracy** — 권위수준 적합도
- **Recall@K** — 기대 청크 포함률
- **MRR** — 첫 관련 청크의 역순위

In [14]:
test_cases = load_test_cases("eval/test_cases.json")
print(f"Loaded {len(test_cases)} test cases\n")

lines = ["| ID | Type | Query | Expected Stds | #Chunks | #XRefs |",
         "|---|---|---|---|---|---|"]
for tc in test_cases:
    short_q = tc.query if len(tc.query) <= 30 else tc.query[:27] + "..."
    stds = ", ".join(tc.expected_standards) if tc.expected_standards else "-"
    lines.append(
        f"| {tc.id} | {tc.query_type} | {short_q} | {stds} | {len(tc.expected_chunks)} | {len(tc.requires_cross_refs)} |"
    )
display(Markdown("\n".join(lines)))

Loaded 20 test cases



| ID | Type | Query | Expected Stds | #Chunks | #XRefs |
|---|---|---|---|---|---|
| tc01_simple_factual | normative | 유형자산의 감가상각 방법에는 어떤 것이 있는가? | K-IFRS 1016 | 1 | 0 |
| tc02_scope_judgment | normative | 리스의 정의는 무엇인가? | K-IFRS 1116 | 0 | 0 |
| tc03_cross_ref_intra | normative | 금융자산의 분류 기준은 무엇인가? | K-IFRS 1109 | 2 | 1 |
| tc04_cross_ref_inter | normative | 수익 인식의 5단계 모형을 설명하라 | K-IFRS 1115 | 0 | 0 |
| tc05_citation_direct | citation | 제1016호 문단 62 | K-IFRS 1016 | 1 | 0 |
| tc06_interpretive | interpretive | 왜 IFRS는 공정가치 측정을 요구하는가? | K-IFRS 1113 | 0 | 0 |
| tc07_example | example | 리스 회계처리 사례를 보여줘 | K-IFRS 1116 | 0 | 0 |
| tc08_multi_standard | comparative | 충당부채와 우발부채의 차이는? | K-IFRS 1037 | 0 | 0 |
| tc09_recognition | normative | 무형자산의 인식 요건은? | K-IFRS 1038 | 1 | 0 |
| tc10_measurement | normative | 재고자산의 측정 방법은? | K-IFRS 1002 | 0 | 0 |
| tc11_disclosure | normative | 관계기업 투자의 공시 사항은? | K-IFRS 1028 | 0 | 0 |
| tc12_ag_specific | normative | 수익 인식에서 본인과 대리인 판단 기준은? | K-IFRS 1115 | 0 | 4 |
| tc13_impairment | normative | 자산손상의 손상차손 인식 요건은? | K-IFRS 1036 | 0 | 0 |
| tc14_consolidation | normative | 연결재무제표 작성 시 지배력 판단 기준은? | K-IFRS 1110 | 0 | 0 |
| tc15_korean_specific | normative | K-IFRS에서 영업이익은 어떻게 표시하는가? | K-IFRS 1001 | 0 | 0 |
| tc16_tax_accounting | normative | 이연법인세의 인식 요건은? | K-IFRS 1012 | 0 | 0 |
| tc17_insurance | normative | 보험계약 측정의 기본 구성요소는? | K-IFRS 1117 | 0 | 0 |
| tc18_fx_effect | comparative | 기능통화와 표시통화의 차이는? | K-IFRS 1021 | 0 | 0 |
| tc19_business_combination | normative | 사업결합에서 영업권은 어떻게 측정하는가? | K-IFRS 1103 | 0 | 0 |
| tc20_conceptual_framework | normative | 재무보고 개념체계에서 자산의 정의는? | 재무보고 개념체계 | 0 | 0 |

In [15]:
eval_results: list[EvalResult] = []

for i, tc in enumerate(test_cases, 1):
    print(f"[{i}/{len(test_cases)}] {tc.id}: {tc.query[:40]}...", end=" ")
    t0 = time.time()
    try:
        docs, _ = run_pipeline(tc.query, client, embeddings, reranker, verbose=False)
        result = evaluate_retrieval(tc, docs)
        eval_results.append(result)
        print(f"OK ({time.time()-t0:.1f}s) DRM={result.drm:.2f} Auth={result.authority_accuracy:.2f}")
    except Exception as e:
        print(f"ERROR: {e}")
        eval_results.append(EvalResult(
            test_id=tc.id, query=tc.query,
            drm=1.0, xref_coverage=0.0, authority_accuracy=0.0,
            recall_at_k=-1.0, mrr=0.0, retrieved_count=0,
        ))

print_eval_summary(eval_results)

[1/20] tc01_simple_factual: 유형자산의 감가상각 방법에는 어떤 것이 있는가?... OK (1.8s) DRM=0.00 Auth=1.00
[2/20] tc02_scope_judgment: 리스의 정의는 무엇인가?... OK (2.1s) DRM=0.15 Auth=0.85
[3/20] tc03_cross_ref_intra: 금융자산의 분류 기준은 무엇인가?... OK (2.0s) DRM=0.50 Auth=1.00
[4/20] tc04_cross_ref_inter: 수익 인식의 5단계 모형을 설명하라... OK (2.2s) DRM=0.31 Auth=0.77
[5/20] tc05_citation_direct: 제1016호 문단 62... OK (0.0s) DRM=1.00 Auth=0.00
[6/20] tc06_interpretive: 왜 IFRS는 공정가치 측정을 요구하는가?... OK (2.2s) DRM=0.21 Auth=0.89
[7/20] tc07_example: 리스 회계처리 사례를 보여줘... OK (1.6s) DRM=1.00 Auth=0.73
[8/20] tc08_multi_standard: 충당부채와 우발부채의 차이는?... OK (2.2s) DRM=0.31 Auth=0.75
[9/20] tc09_recognition: 무형자산의 인식 요건은?... OK (1.7s) DRM=0.00 Auth=1.00
[10/20] tc10_measurement: 재고자산의 측정 방법은?... OK (1.8s) DRM=0.17 Auth=1.00
[11/20] tc11_disclosure: 관계기업 투자의 공시 사항은?... OK (1.6s) DRM=0.67 Auth=0.93
[12/20] tc12_ag_specific: 수익 인식에서 본인과 대리인 판단 기준은?... OK (4.6s) DRM=0.30 Auth=1.00
[13/20] tc13_impairment: 자산손상의 손상차손 인식 요건은?... OK (1.7s) DRM=0.50 Auth=1.00
[

In [16]:
def fmt_metric(val: float, higher_better: bool = True, threshold: float = 0.7) -> str:
    if val < 0:
        return "N/A"
    good = (val >= threshold) if higher_better else (val <= (1.0 - threshold))
    icon = "O" if good else "X"
    return f"{val:.2f} {icon}"

lines = [
    "| Test ID | DRM (low=good) | XRef Coverage | Authority Acc | Recall@K | MRR |",
    "|---|---|---|---|---|---|",
]
for r in eval_results:
    lines.append(
        f"| {r.test_id} "
        f"| {fmt_metric(r.drm, higher_better=False, threshold=0.3)} "
        f"| {fmt_metric(r.xref_coverage)} "
        f"| {fmt_metric(r.authority_accuracy)} "
        f"| {fmt_metric(r.recall_at_k)} "
        f"| {fmt_metric(r.mrr)} |"
    )
display(Markdown("\n".join(lines)))

| Test ID | DRM (low=good) | XRef Coverage | Authority Acc | Recall@K | MRR |
|---|---|---|---|---|---|
| tc01_simple_factual | 0.00 O | 1.00 O | 1.00 O | 1.00 O | 0.50 X |
| tc02_scope_judgment | 0.15 O | 1.00 O | 0.85 O | N/A | 1.00 O |
| tc03_cross_ref_intra | 0.50 O | 1.00 O | 1.00 O | 0.00 X | 0.00 X |
| tc04_cross_ref_inter | 0.31 O | 1.00 O | 0.77 O | N/A | 1.00 O |
| tc05_citation_direct | 1.00 X | 0.00 X | 0.00 X | 0.00 X | 0.00 X |
| tc06_interpretive | 0.21 O | 1.00 O | 0.89 O | N/A | 0.33 X |
| tc07_example | 1.00 X | 1.00 O | 0.73 O | N/A | 0.00 X |
| tc08_multi_standard | 0.31 O | 1.00 O | 0.75 O | N/A | 0.50 X |
| tc09_recognition | 0.00 O | 1.00 O | 1.00 O | 0.00 X | 0.00 X |
| tc10_measurement | 0.17 O | 1.00 O | 1.00 O | N/A | 1.00 O |
| tc11_disclosure | 0.67 O | 1.00 O | 0.93 O | N/A | 1.00 O |
| tc12_ag_specific | 0.30 O | 0.00 X | 1.00 O | N/A | 0.50 X |
| tc13_impairment | 0.50 O | 1.00 O | 1.00 O | N/A | 1.00 O |
| tc14_consolidation | 0.62 O | 1.00 O | 1.00 O | N/A | 0.33 X |
| tc15_korean_specific | 0.67 O | 1.00 O | 0.73 O | N/A | 1.00 O |
| tc16_tax_accounting | 0.00 O | 1.00 O | 0.94 O | N/A | 1.00 O |
| tc17_insurance | 0.00 O | 1.00 O | 0.69 X | N/A | 1.00 O |
| tc18_fx_effect | 0.17 O | 1.00 O | 0.33 X | N/A | 1.00 O |
| tc19_business_combination | 0.77 X | 1.00 O | 0.69 X | N/A | 0.25 X |
| tc20_conceptual_framework | 0.79 X | 1.00 O | 0.93 O | N/A | 0.12 X |

## 11. Deep-dive: Problematic Cases

DRM이 높거나 Authority Accuracy가 낮은 worst 케이스를 상세 분석한다.

In [17]:
# Sort by combined badness: high DRM + low authority
sorted_results = sorted(
    eval_results,
    key=lambda r: r.drm - r.authority_accuracy,
    reverse=True,
)

for r in sorted_results[:3]:
    print(f"\n{'='*60}")
    print(f"[{r.test_id}] DRM={r.drm:.2f}, Auth={r.authority_accuracy:.2f}, XRef={r.xref_coverage:.2f}")
    print(f"Query: {r.query}")
    print('='*60)
    docs, log = run_pipeline(r.query, client, embeddings, reranker, verbose=True)
    display_docs(docs[:8], title=f"{r.test_id} - Top 8 Context Docs")


[tc05_citation_direct] DRM=1.00, Auth=0.00, XRef=0.00
Query: 제1016호 문단 62
  [Route] type=citation, k=5, rerank_n=5, boost=False
  [Retrieve] SKIP vector search, direct IDs: ['KIFRS1016_main_62']
  [Retrieve] 0 docs | {}
  [Rerank] 0 -> 0 docs
  [XRef] +0 docs | total=0
  [Terms] +0 glossary | total=0



[tc07_example] DRM=1.00, Auth=0.73, XRef=1.00
Query: 리스 회계처리 사례를 보여줘
  [Route] type=example, k=20, rerank_n=10, boost=False
  [Retrieve] 20 docs | {'ie': 20}
  [Rerank] 20 -> 10 docs
  [XRef] +3 docs | total=13
  [Terms] +2 glossary | total=15



[tc19_business_combination] DRM=0.77, Auth=0.69, XRef=1.00
Query: 사업결합에서 영업권은 어떻게 측정하는가?
  [Route] type=normative, k=30, rerank_n=10, boost=True
  [Retrieve] 30 docs | {'main': 16, 'ag': 14}
  [Rerank] 30 -> 10 docs
  [Boost] applied (factor=0.85)
  [XRef] +1 docs | total=11
  [Terms] +2 glossary | total=13


## 12. Cleanup

In [18]:
client.close()
print("Qdrant connection closed")

Qdrant connection closed
